In [0]:
%pip install -qqqq -U mlflow[genai,databricks] databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("experiment_name", "", "Experiment Name")
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

In [0]:
import os
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

default_warehouse = next(
    (
        wh
        for wh in w.warehouses.list()
        if "Serverless Starter Warehouse" in wh.name and wh.enable_serverless_compute
    ),
    None,
)
default_warehouse_id = default_warehouse.id if default_warehouse else None
print(f"{default_warehouse_id=}")

# Specify the ID of a SQL warehouse you have access to.
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = default_warehouse_id

## [Step 1: Set Up the Experiment](https://mlflow.org/cookbook/genie-evaluation-judges/#step-1-set-up-the-experiment)

In [0]:
import mlflow
from mlflow.entities import Feedback
from mlflow.genai.scorers import (
    Guidelines,
    RelevanceToQuery,
    RetrievalGroundedness,
    Safety,
    scorer,
)

mlflow.set_experiment(EXPERIMENT_NAME)

## [Step 2: Define Built-in LLM Judges](https://mlflow.org/cookbook/genie-evaluation-judges/#step-2-define-llm-judges)

In [0]:
relevance = RelevanceToQuery(model="databricks:/databricks-qwen35-122b-a10b")
safety = Safety(model="databricks:/databricks-gpt-5-6-luna")
groundedness = RetrievalGroundedness(model="databricks:/databricks-claude-sonnet-5")

## [Step 3: Define Custom Guidelines Judges](https://mlflow.org/cookbook/genie-evaluation-judges/#step-3-define-custom-judges)

In [0]:
response_quality = Guidelines(
    name="genie_response_quality",
    guidelines=[
        "The response must directly address the user's data question "
        "rather than giving a vague or generic reply.",
        "If SQL was generated, the response must include a data-driven "
        "answer, not just echo the SQL query back.",
        "The response must not say 'I cannot answer' when the question "
        "is about data that should be available in the tables.",
    ],
    model="databricks:/databricks-claude-sonnet-5",
)

sql_quality = Guidelines(
    name="genie_sql_quality",
    guidelines=[
        "If SQL is present, it must use appropriate aggregation "
        "functions (SUM, COUNT, AVG) matching the user's intent.",
        "The SQL must include appropriate WHERE clauses to filter "
        "data as the user requested.",
        "The SQL must not use SELECT * on large tables without a "
        "LIMIT or specific filter.",
    ],
    model="databricks:/databricks-claude-sonnet-5",
)

## [Step 4: Define Code-Based Scorers](https://mlflow.org/cookbook/genie-evaluation-judges/#step-4-define-code-based-scorers)

In [0]:
@scorer
def has_response(outputs) -> Feedback:
    """Check if Genie returned a text response."""
    resp = outputs.get("response") if isinstance(outputs, dict) else None
    if resp and len(str(resp).strip()) > 0:
        return Feedback(value="yes", rationale=f"{len(resp)} chars")
    return Feedback(value="no", rationale="No text response")


@scorer
def no_error(outputs) -> Feedback:
    """Check that the interaction completed without errors."""
    err = outputs.get("error") if isinstance(outputs, dict) else None
    if err and str(err).strip():
        return Feedback(value="no", rationale=f"Error: {str(err)[:200]}")
    return Feedback(value="yes", rationale="No errors")

## [Step 5: Run Evaluation](https://mlflow.org/cookbook/genie-evaluation-judges/#step-5-run-evaluation)

In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
traces_df = mlflow.search_traces(
    locations=[experiment.experiment_id],
    order_by=["timestamp DESC"],
    max_results=100,
)
print(f"Found {len(traces_df)} traces to evaluate")

eval_results = mlflow.genai.evaluate(
    data=traces_df,
    scorers=[
        relevance,
        safety,
        groundedness,
        response_quality,
        sql_quality,
        has_response,
        no_error,
    ],
)